In [1]:
# ---------------------------------------------------------------------------
# Jugador
# ---------------------------------------------------------------------------
class Jugador:
    HP_MAXIMA = 50

    def __init__(self, nombre, nivel=1, clase=None):
        self.nombre = nombre
        self.nivel = nivel
        self.experiencia = 0
        self.oro = 0
        self.hp_maxima = self.HP_MAXIMA
        self.hp = self.hp_maxima
        self.clase = clase if clase is not None else Aprendiz()
        self.clan = None
        self.inventario = []      # Equipos adquiridos en la tienda

    # Experiencia necesaria para pasar del nivel actual al siguiente
    def exp_para_subir(self):
        return 50 * self.nivel

    # Agrega experiencia (ya con modificadores aplicados) y sube de nivel si corresponde
    def ganar_experiencia(self, cantidad):
        if cantidad > 0:
            self.experiencia += cantidad
            while self.experiencia >= self.exp_para_subir():
                self.experiencia -= self.exp_para_subir()
                self._subir_nivel()

    def _subir_nivel(self):
        self.nivel += 1
        self.curar(self.clase.curacion_al_subir_nivel(self.hp_maxima))

    def curar(self, cantidad):
        if cantidad > 0:
            self.hp = min(self.hp_maxima, self.hp + cantidad)

    # Recibe daño base; la clase puede reducirlo. Devuelve el daño realmente recibido
    def recibir_dano(self, base):
        dano = self.clase.modificar_dano(base)
        self.hp = max(0, self.hp - dano)
        return dano

    # Aplica una recompensa {"experiencia", "oro", "vida"} usando los modificadores de la clase
    def aplicar_recompensa(self, recompensa):
        if recompensa is None:
            return False
        self.ganar_experiencia(self.clase.modificar_exp(recompensa["experiencia"]))
        self.oro += self.clase.modificar_oro(recompensa["oro"])
        vida = recompensa["vida"]
        if vida > 0:
            self.curar(vida)
        elif vida < 0:
            self.recibir_dano(-vida)
        return True

    # Completa una tarea (o hábito positivo). Devuelve False si ya estaba completada hoy
    def completar_tarea(self, tarea):
        return self.aplicar_recompensa(tarea.completar())

    # Marca un hábito como "Positivo" o "Negativo"
    def marcar_habito(self, habito, forma="Positivo"):
        if forma == "Positivo":
            recompensa = habito.marcar_positivo()
        elif forma == "Negativo":
            recompensa = habito.marcar_negativo()
        else:
            raise ValueError("La forma debe ser 'Positivo' o 'Negativo'.")
        return self.aplicar_recompensa(recompensa)

    # Muestra los datos del jugador
    def mostrar_datos(self):
        print(f"Jugador: {self.nombre}")
        print(f"Clase: {self.clase.nombre}")
        print(f"Nivel: {self.nivel}")
        print(f"Experiencia: {self.experiencia}/{self.exp_para_subir()}")
        print(f"Vida: {self.hp}/{self.hp_maxima}")
        print(f"Oro: {self.oro}")
        print(f"Clan: {self.clan.nombre if self.clan else 'Sin clan'}")
        print(f"Inventario: {', '.join(i.nombre for i in self.inventario) or 'Vacío'}")


# ---------------------------------------------------------------------------
# Clanes
# ---------------------------------------------------------------------------
# Representa un clan de jugadores (base de ClanAbierto y ClanPrivado)
class Clan:
    def __init__(self, nombre, capacidad=10):
        self.nombre = nombre
        self.capacidad = capacidad
        self.miembros = []

    def tiene_espacio(self):
        return len(self.miembros) < self.capacidad

    # Agrega un jugador si hay espacio, no es miembro y no pertenece a otro clan
    def agregar_miembro(self, jugador):
        if not self.tiene_espacio() or jugador in self.miembros or jugador.clan is not None:
            return False
        self.miembros.append(jugador)
        jugador.clan = self
        return True

    # Muestra los miembros del clan
    def mostrar_miembros(self):
        print(f"Clan: {self.nombre}")
        for jugador in self.miembros:
            print(f"- {jugador.nombre} (Nivel {jugador.nivel}, {jugador.clase.nombre})")


# Representa un clan al que cualquiera puede solicitar ingreso
class ClanAbierto(Clan):
    # Permite ingresar mientras haya espacio
    def unirse(self, jugador):
        return self.agregar_miembro(jugador)


# Representa un clan que requiere invitación
class ClanPrivado(Clan):
    def __init__(self, nombre, capacidad=10):
        super().__init__(nombre, capacidad)
        self.invitados = []

    # Registra una invitación para un jugador
    def invitar(self, jugador):
        if jugador not in self.miembros and jugador not in self.invitados:
            self.invitados.append(jugador)

    # En un clan privado solo se ingresa con invitación
    def agregar_miembro(self, jugador):
        return self.aceptar_invitacion(jugador)

    # Permite ingresar a un jugador invitado
    def aceptar_invitacion(self, jugador):
        if jugador not in self.invitados:
            return False
        if super().agregar_miembro(jugador):
            self.invitados.remove(jugador)
            return True
        return False


In [ ]:
# Clase general ClasePersonaje y sus cinco hijas: Aprendiz, Guerrero, Mago, Sanador y Picaro.
# Cada clase modifica la EXP, el oro y el daño que recibe el jugador.

# Creamos _redondear para evitar errores al redondear con round()
# round(2.5) = 2 -> No es conveniente para nuestro escenario
def _redondear(valor):
    return int(valor + 0.5)

class ClasePersonaje:
    def __init__(self, nombre, descripcion, mult_exp=1.0, mult_oro=1.0,
                 reduccion_dano=0.0, curacion_nivel=0.5):
        self.nombre = nombre
        self.descripcion = descripcion
        self.mult_exp = mult_exp                # 1.0 = sin bonificación
        self.mult_oro = mult_oro                # 1.0 = sin bonificación
        self.reduccion_dano = reduccion_dano    # fracción del daño que se evita
        self.curacion_nivel = curacion_nivel    # fracción de vida que se restaura al subir de nivel

    def __str__(self):
        return f"{self.nombre}: {self.descripcion}"
    
    def modificar_exp(self, base):
        return _redondear(base * self.mult_exp)

    def modificar_oro(self, base):
        return _redondear(base * self.mult_oro)

    def modificar_dano(self, base):
        # Devuelve el daño final que recibe el jugador 
        if base <= 0:                # sin daño (o valor inválido)
            return 0
        # (1 - reduccion_dano) es la fracción de daño que recibe el jugador
        # max(1, ...) asegura que, si hubo daño, se reciba al menos 1
        return max(1, _redondear(base * (1 - self.reduccion_dano)))

    def curacion_al_subir_nivel(self, hp_maxima):
        return _redondear(hp_maxima * self.curacion_nivel)

# Clase hijas 
class Aprendiz(ClasePersonaje):
    def __init__(self):
        super().__init__("Aprendiz", "Clase inicial, sin bonificaciones")


class Guerrero(ClasePersonaje):
    def __init__(self):
        super().__init__("Guerrero", "Muy resistente contra el daño", reduccion_dano=0.30)


class Mago(ClasePersonaje):
    def __init__(self):
        super().__init__("Mago", "Sube de nivel más rápido", mult_exp=1.25)


class Sanador(ClasePersonaje):
    def __init__(self):
        super().__init__("Sanador", "Restaura su salud al subir de nivel", curacion_nivel=1.0)


class Picaro(ClasePersonaje):
    def __init__(self):
        super().__init__("Picaro", "Obtiene oro más rápido", mult_oro=1.25)

In [ ]:
class ItemTienda:
    # Atributo de clase autoincremental: cada ítem recibe un código único
    _contador_codigo = 1

    def __init__(self, nombre, precio, nivel_requerido, descripcion):
        # Se valida antes de asignar el código para no "gastar" códigos en ítems inválidos
        if precio < 0:
            raise ValueError("El precio no puede ser negativo.")
        if nivel_requerido < 1:
            raise ValueError("El nivel requerido debe ser al menos 1.")
        self.codigo = ItemTienda._contador_codigo
        ItemTienda._contador_codigo += 1
        self.nombre = nombre
        self.precio = precio
        self.nivel_requerido = nivel_requerido
        self.descripcion = descripcion

    # Aplica el efecto del ítem al jugador. Devuelve True si se aplicó (y por tanto se cobra)
    def aplicar(self, jugador):
        raise NotImplementedError("Cada tipo de ítem define su propio efecto.")


class Equipo(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, bonificacion):
        if bonificacion <= 0:
            raise ValueError("La bonificación debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.bonificacion = bonificacion    # puntos que aumentan la hp_maxima del jugador

    def aplicar(self, jugador):
        # Se guarda en el inventario y mejora la vida máxima (sin repetir el mismo equipo)
        if self in jugador.inventario:
            print(f"Ya tienes equipado: {self.nombre}.")
            return False
        jugador.inventario.append(self)
        jugador.hp_maxima += self.bonificacion
        print(f"Has equipado: {self.nombre}. Bonificación: +{self.bonificacion} HP máxima")
        return True


class Pocion(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, hp_restaurada):
        if hp_restaurada <= 0:
            raise ValueError("La vida restaurada debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.hp_restaurada = hp_restaurada

    def aplicar(self, jugador):
        # Cura al jugador respetando su hp_maxima (usa Jugador.curar)
        if jugador.hp >= jugador.hp_maxima:
            print("Tu salud ya está al máximo; no se usó la poción.")
            return False
        hp_antes = jugador.hp
        jugador.curar(self.hp_restaurada)
        print(f"Te has curado {jugador.hp - hp_antes} HP. Salud actual: {jugador.hp}/{jugador.hp_maxima}")
        return True


class Recompensa(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, creada_por_usuario):
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.creada_por_usuario = creada_por_usuario

    def aplicar(self, jugador):
        # Solo se registra el premio real
        print(f"Recompensa canjeada: {self.nombre}. ¡Disfruta tu premio!")
        return True


class Tienda:
    def __init__(self):
        self.items = []    # Lista de objetos ItemTienda

    def agregar_item(self, item):
        if not isinstance(item, ItemTienda):
            raise TypeError("Solo se pueden agregar objetos de tipo ItemTienda.")
        if item not in self.items:
            self.items.append(item)

    def mostrar_tabla(self):
        print("\n--- OBJETOS DE LA TIENDA ---")
        if not self.items:
            print("La tienda está vacía.")
            return
        for item in self.items:
            print(f"[{item.codigo}] {item.nombre} | Nivel req: {item.nivel_requerido} | Precio: {item.precio} oro")
            print(f"    Desc: {item.descripcion}")

    def buscar_por_codigo(self, codigo):
        for item in self.items:
            if item.codigo == codigo:
                return item
        return None

    # Compra un ítem: valida existencia, nivel y oro; cobra solo si el ítem se aplicó
    def comprar(self, jugador, codigo):
        item = self.buscar_por_codigo(codigo)
        if item is None:
            print(f"No existe un ítem con código {codigo}.")
            return False
        if jugador.nivel < item.nivel_requerido:
            print(f"Necesitas nivel {item.nivel_requerido} para comprar {item.nombre}.")
            return False
        if jugador.oro < item.precio:
            print(f"Oro insuficiente para {item.nombre}: tienes {jugador.oro} y cuesta {item.precio}.")
            return False
        if not item.aplicar(jugador):
            return False
        jugador.oro -= item.precio
        print(f"Compra realizada: {item.nombre} (-{item.precio} oro). Oro restante: {jugador.oro}")
        return True

In [ ]:
from datetime import datetime
import calendar
import re
FORMATO_FECHA = "%Y-%m-%d"

def hoy():
    return datetime.now().date()

class Tarea():
    """Representa una tarea completada o sin completar de un Jugador.
    
    Atributos:
        titulo: El nombre de la tarea.
        dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre la tarea.
        historial: Lista de fechas en las que la tarea fue completada.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
    """
    RECOMPENSAS = {"Facil": {"experiencia":6, "oro": 1, "vida":0},
                   "Intermedio": {"experiencia":8, "oro":1, "vida":1},
                   "Dificil": {"experiencia":12, "oro": 3, "vida":4}}
    
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None):
        if dificultad in self.RECOMPENSAS.keys():
            self.titulo = titulo
            self.dificultad = dificultad
            self.descripcion = descripcion
            self.historial = []
            self.fecha_creacion = str(hoy())
        else:
            raise ValueError(f"Se añadio una dificultad no válida.\n Debe ser una entre {list(self.RECOMEPNSAS.keys())}")

    def editar_tarea(self, nuevo_titulo = None, nueva_dificultad = None, nueva_descripcion = None):
        if nuevo_titulo is not None:
            self.titulo = nuevo_titulo
        if nueva_dificultad is not None:
            if nueva_dificultad not in self.RECOMPENSAS.keys():
                raise ValueError(f"Se añadio una dificultad no válida.\n Debe ser una entre {list(self.RECOMPENSAS.keys())}")
            self.dificultad = nueva_dificultad
        if nueva_descripcion is not None:
            self.descripcion = nueva_descripcion
    
    def calcular_recompensa(self):
        return dict(self.RECOMPENSAS[self.dificultad])

    def completar(self):
        hoy1 = str(hoy())
        if hoy1 not in self.historial:
            self.historial.append(hoy1)
            return self.calcular_recompensa()
        return None
        
    def __repr__(self):
        recompensa = self.calcular_recompensa()
        return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa ---\n| Experiencia: {recompensa["experiencia"]} | Oro: {recompensa["oro"]} | Vida: {recompensa["vida"]} |"

class Habito(Tarea):
    '''Representa un Habito positivo o negativo que forma un Jugador

    Atributos:
        titulo: El nombre del habito.
        dificultad: El nivel de dificultad del habito ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre el habito.
        tipo_habito: El tipo de habito que representa el objeto ('Positivo', 'Negativo' o 'Ambos').
        hitorial_positivo: Lista de fechas en la que un habito positivo fue completado.
        historial_negativo: Lista de fechas en la que un habito negativo fue completado.
        historial: Lista de fechas en las que un habito fue completada.
        contador_positivo: Numero de dias consecutivos realizando un habito positivo.
        contador_negativo: Numero de dias consecutivos realizando un habito negativo.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Habito.
    '''
    TIPOS_HABITO = ["Positivo", "Negativo", "Ambos"]
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None, tipo_habito = "Ambos"):
        if tipo_habito not in self.TIPOS_HABITO:
            raise ValueError(f"Se añadio un tipo de hábito no válido.\n Debe ser uno entre {self.TIPOS_HABITO}")
        super().__init__(titulo, dificultad, descripcion)
        self.tipo_habito = tipo_habito
        self.contador_positivo = 0
        self.historial_positivo = []
        self.contador_negativo = 0
        self.historial_negativo = []

    def calcular_recompensa(self, forma):
        if forma not in ["Positivo", "Negativo"]:
            raise ValueError("La forma debe ser 'Positivo' o 'Negativo'.")
        recompensa = super().calcular_recompensa()
        if forma == "Negativo":
            recompensa["experiencia"] = recompensa["experiencia"]//2
            recompensa["oro"] = 0
            recompensa["vida"] = -1 * recompensa["vida"] - 2
        return recompensa

    @staticmethod
    def _calcular_racha(historial, contador):
        if not historial:
            return 1
        ultima = datetime.strptime(historial[-1], FORMATO_FECHA).date()
        dias = (hoy()-ultima).days
        if dias == 0:
            return contador
        if dias == 1:
            return contador + 1
        return 1

    def marcar_positivo(self):
        if self.tipo_habito in ["Positivo", "Ambos"]:
            self.contador_positivo = self._calcular_racha(self.historial_positivo, self.contador_positivo)
            hoy1 = str(hoy())
            self.historial_positivo.append(hoy1)
            self.historial.append(hoy1)
            return self.calcular_recompensa("Positivo")
        else:
            raise ValueError("El hábito no se completo. El Hábito no es del tipo <Hábito Positivo>.")
    
    def marcar_negativo(self):
        if self.tipo_habito in ["Negativo", "Ambos"]:
            self.contador_negativo = self._calcular_racha(self.historial_negativo, self.contador_negativo)
            hoy1 = str(hoy())
            self.historial_negativo.append(hoy1)
            self.historial.append(hoy1)
            return self.calcular_recompensa("Negativo")
        else:
            raise ValueError("El hábito no se completo. El Hábito no es del tipo <Hábito Negativo>.")
        
    def completar(self):
        if self.tipo_habito == "Positivo":
            return self.marcar_positivo()
        elif self.tipo_habito == "Negativo":
            return self.marcar_negativo()
        elif self.tipo_habito == "Ambos":
            return [self.marcar_positivo(), self.marcar_negativo()]

    def __repr__(self):
        if self.tipo_habito in ["Positivo", "Negativo"]:
            recompensa = self.calcular_recompensa(self.tipo_habito)
            return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Tipo de Habito: {self.tipo_habito} |Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa ---\n| Experiencia: {recompensa["experiencia"]} | Oro: {recompensa["oro"]} | Vida: {recompensa["vida"]} |"
        else:
            recompensa_positiva = self.calcular_recompensa("Positivo")
            recompensa_negativa = self.calcular_recompensa("Negativo")
            return f" ---------- Información sobre la Tarea ---------- \n| Título: {self.titulo} | Dificultad: {self.dificultad} | Descripcion: {self.descripcion} | Tipo de Habito: {self.tipo_habito} |Fecha de Creación: {self.fecha_creacion} | Nro. de Fechas Cumplidas: {len(self.historial)} | \n\n--- Información sobre la Recompensa Positiva ---\n| Experiencia: {recompensa_positiva["experiencia"]} | Oro: {recompensa_positiva["oro"]} | Vida: {recompensa_positiva["vida"]} | \n\n--- Información sobre la Recompensa Negativa ---\n| Experiencia: {recompensa_negativa["experiencia"]} | Oro: {recompensa_negativa["oro"]} | Vida: {recompensa_negativa["vida"]} |"

class Diaria(Tarea):
    """Representa una tarea de un Jugador que se debe de cumplir una vez al día.
        
        Atributos:
            titulo: El nombre de la tarea.
            dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
            descripcion: Detalles adicionales sobre la tarea.
            historial: Lista de fechas en las que la tarea fue completada.
            fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
            racha: Número consecutivos de días en el que se completo la tarea Diaria.
            mejor_racha: Mayor número de días consecutivos en el que se completo la tarea Diaria.
            fecha_inicio_racha: La fecha en la que inicio la racha actual.
    """
    def __init__(self, titulo:str, dificultad:str, descripcion:str = None):
        super().__init__(titulo, dificultad, descripcion)
        self.racha = 0
        self.mejor_racha = 0
        self.fecha_inicio_racha = None

    def completar(self):
        self.racha += 1
        if len(self.historial) == 0:
            self.fecha_inicio_racha = str(hoy())
        if len(self.historial) >= 2:
            if (datetime.strptime(self.historial[-1], FORMATO_FECHA) - datetime.strptime(self.historial[-2], FORMATO_FECHA)).days > 1:
                self.fecha_inicio_racha = str(hoy())
                self.racha = 1
        return super().completar()

    def fallar(self):
        if self.racha >= self.mejor_racha:
            self.mejor_racha = self.racha
        self.racha = 0
        recompensa = self.calcular_recompensa()
        recompensa["experiencia"] = 0
        recompensa["oro"] = 0
        recompensa["vida"] = -10
        return recompensa

class Pendiente(Tarea):
    """Representa una tarea pendiente con fecha limite <fecha_limite> de un Jugador.
    
    Atributos:
        titulo: El nombre de la tarea.
        dificultad: El nivel de dificultad de la tarea ('Facil', 'Intermedio' o 'Dificil').
        descripcion: Detalles adicionales sobre la tarea.
        historial: Lista de fechas en las que la tarea fue completada.
        fecha_creacion: Fecha en la que se creo un objeto de la clase Tarea.
        fecha_limite: Fecha máxima para completar la tarea. Pasada la fecha, la recompensa disminuira.
        completada: Estado de completado de la tarea pendiente.
        vencida: Estado de vencimiento de la tarea pendiente.
    """
    def __init__(self, titulo:str, dificultad:str, fecha_limite:str, descripcion:str = None):
        max = datetime.strptime(fecha_limite, FORMATO_FECHA).date()
        super().__init__(titulo, dificultad, descripcion)
        if (max - hoy()).days >= 0:
            self.fecha_limite = fecha_limite
        else:
            raise ValueError("La fecha limite debe ser una igual o posterior a la de hoy.")
        self.completada = False
        self.vencida = False
        
    def esta_vencida(self):
        self.vencida = True

    def calcular_recompensa(self):
        recompensa = super().calcular_recompensa()
        if self.vencida:
            recompensa["experiencia"] -= 4
            recompensa["oro"] -= 1
            recompensa["vida"] = 0
        return recompensa

class Calendario_Racha():
    """Representa el calendario de las rachas del Jugador en una Tarea.
        
        Atributos:
            tarea: Tarea de la cual se mostrara su calendario.
            inicio: Fecha de creacion de Tarea.
    """
    def __init__(self, tarea):
        if not issubclass(type(tarea), Tarea):
            raise ValueError("El argumento <tarea> debe ser de la clase Tarea")
        self.tarea = tarea
        self.inicio = tarea.fecha_creacion

    def mostrar(self, tipo_historial = None):
        if not isinstance(self.tarea, Habito) and tipo_historial is not None:
            raise ValueError(f"El parametro <tipo_historial> solo debe ser ingresado para instancias de Habito.")
        elif isinstance(self.tarea, Habito) and tipo_historial is None:
            raise ValueError(f"El parametro <tipo_historial> debe completarse para instancias de Habito.")
        tipos_historial = ["historial_positivo", "historial_negativo", "historial_completo"]
        if tipo_historial is None:
            tipo_historial = "historial_completo"
        if tipo_historial not in tipos_historial:
            raise ValueError(f"El tipo de historial ingresado no es un tipo válido para la clase Habito. El <tipo_historial> debe ser uno entre {tipos_historial}")

        if tipo_historial == "historial_completo":
            fechas_str = self.tarea.historial
        elif tipo_historial == "historial_positivo":
            fechas_str = self.tarea.historial_positivo
        elif tipo_historial == "historial_negativo":
            fechas_str = self.tarea.historial_negativo

        if len(fechas_str) == 0:
            anio = int(self.inicio[:4])
            mes = int(self.inicio[5:7])
            print(calendar.month(anio, mes))
            return

        dict_fechas = {}
        for fecha in fechas_str:
            fecha_dt = datetime.strptime(fecha, FORMATO_FECHA)
            key_mes = (fecha_dt.year, fecha_dt.month)
            if key_mes not in dict_fechas.keys():
                dict_fechas[key_mes] = []
            dict_fechas[key_mes].append(str(fecha_dt.day))

        ults_tres_meses = sorted(dict_fechas.keys())[-3:]

        for anio_mes in ults_tres_meses:
            anio, mes = anio_mes
            dias = dict_fechas[anio_mes]
            lineas = calendar.month(anio, mes).split("\n")

            for i in range(2, len(lineas)):
                for dia in dias:
                    patron = rf"\b{dia}\b"
                    reemplazo = f"\033[91m{dia}\033[0m"
                    lineas[i] = re.sub(patron, reemplazo, lineas[i])
            print("\n".join(lineas))